# Component retrieval on a free Kaggle GPU
This is a new experimental profile. The preserved first Qwen run failed its coverage gate. This notebook runs the prospectively selected component policy and matched whole-query comparator, while preserving independent human screening and extraction. No paid inference, provider key or local Mac model is used.

Upload the locally prepared **private** input bundle with exactly `job.json`, `component_core.py`, `qwen_kaggle_environment.py`, and `qwen_components_kaggle_runner.py`. Set Internet On and a CUDA GPU offered by your account; check available quota. Save a finite notebook version with the interactive draft off. Keep inputs and outputs private. Sources and expected-answer truth remain separate: only questions/components and source passages cross this boundary.

In [ ]:
import os
from pathlib import Path
BUNDLE = Path('/kaggle/input/YOUR_PRIVATE_DATASET')
JOB_PATH = BUNDLE / 'job.json'
CORE_PATH = BUNDLE / 'component_core.py'
HELPER_PATH = BUNDLE / 'qwen_kaggle_environment.py'
RUNNER_PATH = BUNDLE / 'qwen_components_kaggle_runner.py'
JOB_SHA256 = 'PASTE_TRUSTED_LOCAL_EXPORT_SHA256_HERE'
OUTPUT_PATH = Path('/kaggle/working/components-results.json')
CHECKPOINT_PATH = Path('/kaggle/working/components-checkpoint.json')
AUDIT_PATH = Path('/kaggle/working/components-environment-audit.json')
INSTALLER_REPORT_PATH = Path('/kaggle/working/components-installer-report.json')
os.environ['HF_HOME'] = '/kaggle/temp/components-huggingface'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

## Verify executable bytes before loading helpers
Keep the job hash from the local export. This cell checks the job and every executable against those trusted bindings before executing either helper. It does not install packages or load models in the notebook's shared environment.

In [ ]:
import hashlib
import importlib.util
import json
def unique_pairs(items):
    result = {}
    for name, value in items:
        if name in result: raise ValueError('Duplicate JSON key')
        result[name] = value
    return result
def reject_constant(value):
    raise ValueError('Nonfinite JSON')
assert JOB_PATH.stat().st_size <= 100 * 1024 * 1024
envelope = json.loads(JOB_PATH.read_bytes(), object_pairs_hook=unique_pairs, parse_constant=reject_constant)
assert set(envelope) == {'payload', 'payload_sha256'}
payload_bytes = json.dumps(envelope['payload'], ensure_ascii=False, sort_keys=True, separators=(',', ':'), allow_nan=False).encode('utf-8')
assert hashlib.sha256(payload_bytes).hexdigest() == envelope['payload_sha256'] == JOB_SHA256
executables = {'component_core.py': CORE_PATH, 'qwen_kaggle_environment.py': HELPER_PATH, 'qwen_components_kaggle_runner.py': RUNNER_PATH}
assert set(envelope['payload']['execution_files']) == set(executables)
for name, path in executables.items():
    assert path.name == name and path.resolve().is_relative_to(BUNDLE.resolve())
    body = path.read_bytes()
    assert {'sha256': hashlib.sha256(body).hexdigest(), 'size_bytes': len(body)} == envelope['payload']['execution_files'][name]
spec = importlib.util.spec_from_file_location('verified_qwen_environment', HELPER_PATH)
helper = importlib.util.module_from_spec(spec)
spec.loader.exec_module(helper)
print({'job_sha256': JOB_SHA256, 'questions': len(envelope['payload']['questions']), 'encoded_queries': len(envelope['payload']['query_rows'])})

## Run the isolated installer and fresh GPU worker
A temporary default virtual environment supplies only the pip installer. Four pinned packages are installed without dependencies into a private temporary overlay. A fresh base-Python worker starts in isolated mode, explicitly prepends that overlay, checks active dependency constraints and actual module paths, and performs a CUDA allocation/operation smoke check before any model weights. Kaggle's global Gradio, Diffusers, transitive packages and CUDA Torch are preserved. This scoped overlay is not complete dependency isolation.

Missing dependencies, incompatible versions, CUDA failures or more than 4,000 distinct rerank pairs stop execution without an automatic upgrade, smaller model, paid fallback or partial success. Model caches and installer/overlay stay outside saved working output. Existing result/checkpoint/audit/report files are preserved; choose new names for another run.

In [ ]:
summary = helper.prepare_environment(JOB_PATH, JOB_SHA256, core_path=CORE_PATH, runner_path=RUNNER_PATH,
                                     output_path=OUTPUT_PATH, checkpoint_path=CHECKPOINT_PATH,
                                     audit_path=AUDIT_PATH, report_path=INSTALLER_REPORT_PATH)
print(json.dumps(summary, sort_keys=True))

## Save evidence and return it to the engine
Keep the printed result payload SHA-256 and the finite saved script version. Save/download the result, checkpoint, environment audit, installer report and first-run log. The worker logs real stage progress and elapsed time. Stop the GPU session when finished.

Import results locally with the original job and trusted job/result hashes. The engine reconstructs all pools, complete score matrices, shared score overlap, selections and own anchors; changed sources, screening states or helper bytes require a new export. Only five unique own passages are displayed; separate pool diagnostics explain selection losses. Candidate grouping and component attribution do not verify support, semantic completeness, report/study linkage or source answerability. The frozen development and untouched confirmation comparisons must pass their declared gates before the human protocol pilot.